In [ ]:
import os
import shutil

#### Functions

#### Constants

In [ ]:
# bucket
str_bucket = os.getcwd().split('/')[4].replace('_', '-')
print(f'Bucket: {str_bucket}')

# step
str_step = os.getcwd().split('/')[-1]
print(f'Step: {str_step}')

# artifact paths
str_pipeline_src = '../03_preprocessing/output/preprocessing_pipeline.joblib'
str_model_src = '../04_model/output/xgboost_model.joblib'
str_feature_cols_src = '../04_model/output/feature_cols.joblib'

# docker image name
str_image_name = str_bucket
print(f'Docker image: {str_image_name}')

# output directory
os.makedirs('output', exist_ok=True)

#### Copy Model Artifacts

Copy the preprocessing pipeline, trained model, and feature column list into the API build context so they can be included in the Docker image.

In [ ]:
# copy artifacts into api directory for docker build context
shutil.copy2(str_pipeline_src, 'preprocessing_pipeline.joblib')
shutil.copy2(str_model_src, 'xgboost_model.joblib')
shutil.copy2(str_feature_cols_src, 'feature_cols.joblib')

print('Artifacts copied:')
for str_file in ['preprocessing_pipeline.joblib', 'xgboost_model.joblib', 'feature_cols.joblib']:
    print(f'  {str_file}')

#### requirements.txt

In [ ]:
%%writefile requirements.txt
fastapi>=0.100.0
uvicorn>=0.23.0
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.3.0
xgboost>=2.0,<3.0
joblib>=1.3.0

#### app.py

The FastAPI application loads the preprocessing pipeline, trained XGBoost model, and feature column list at startup. It exposes a `/predict` endpoint that accepts loan application data as JSON, runs it through the preprocessing pipeline, selects the model features, and returns the predicted probability of default.

In [ ]:
%%writefile app.py
import joblib
import pandas as pd
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

# load artifacts
pipeline = joblib.load('preprocessing_pipeline.joblib')
model = joblib.load('xgboost_model.joblib')
list_feature_cols = joblib.load('feature_cols.joblib')

# define app
app = FastAPI(title='Credit Risk Model API', version='1.0.0')

# define request schema
class LoanApplication(BaseModel):
    loan_id: Optional[int] = None
    origination_date: Optional[str] = None
    dob: Optional[str] = None
    loan_amount: float
    term_months: int
    channel: str
    employment_length_years: Optional[float] = None
    stated_income: Optional[float] = None
    state: Optional[str] = None
    has_prior_loans_with_us: int
    bureau_score: Optional[float] = None
    open_trades: Optional[float] = None
    delinq_12m: Optional[float] = None
    utilization: Optional[float] = None
    inquiries_6m: Optional[float] = None
    public_records: int

# define response schema
class PredictionResponse(BaseModel):
    flt_probability_of_default: float

# health check
@app.get('/health')
def health():
    return {'status': 'healthy'}

# predict endpoint
@app.post('/predict', response_model=PredictionResponse)
def predict(application: LoanApplication):
    # convert to dataframe
    df_input = pd.DataFrame([application.model_dump()])
    # preprocess
    arr_transformed = pipeline.transform(df_input)
    # get column names from pipeline
    list_all_cols = (pipeline.named_steps['column_transformer']
                     .get_feature_names_out().tolist())
    df_transformed = pd.DataFrame(arr_transformed, columns=list_all_cols)
    # select model features
    arr_features = df_transformed[list_feature_cols].values
    # predict
    flt_pred = float(model.predict_proba(arr_features)[:, 1][0])
    return PredictionResponse(flt_probability_of_default=flt_pred)

#### Dockerfile

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# copy artifacts
COPY preprocessing_pipeline.joblib .
COPY xgboost_model.joblib .
COPY feature_cols.joblib .

# copy application
COPY app.py .

# expose port
EXPOSE 8080

# run
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8080"]

#### Build Docker Image

In [ ]:
!docker build -t {str_image_name} .

#### Clean Up

Remove the copied model artifacts from the API directory. They are now baked into the Docker image and no longer needed in the build context.

In [ ]:
# remove copied artifacts
for str_file in ['preprocessing_pipeline.joblib', 'xgboost_model.joblib', 'feature_cols.joblib']:
    os.remove(str_file)
    print(f'Removed: {str_file}')